# Preprocessing audit: `all_removals`

Before building model-ready data: what in `analysis_measurement` is **duplicated**, **wrong**, an **outlier**, **doesn't belong**, or **unevenly sampled**, and how much each fix would cost.

- **Read-only.** The warehouse files are only read (through an in-memory DuckDB, so `warehouse.duckdb` is never opened). Nothing is removed or corrected here.
- Each check becomes a **flag**. Section 8 counts what would remain if each flag were turned into an exclusion. **Nothing is applied**; that is a decision for the next stage.
- Three files are written to `data/processed/preprocessing_audit/all_removals/`: `measurement_flags.parquet`, `pair_features.parquet` and `rule_impact.csv`.

Run on a compute node (about 32 GB): `jupyter nbconvert --to html --execute --ExecutePreprocessor.timeout=-1 notebooks/preprocessing_audit.ipynb`

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from analysis import audit_utils as au
from etl.build_views import DEFAULT_MIN_SAMPLES, DEFAULT_MIN_YEARS
from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.nhdplus import find_huc8_geodatabase
from etl.common.warehouse import warehouse_dir

# Where data/ lives: the repo on the cluster (DAM_REMOVAL_PROJECT_ROOT overrides it, as in the ETL).
DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)
OUTPUT_DIR = DATA_ROOT / "data" / "processed" / "preprocessing_audit" / CASE_STUDY

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE = int(window.get("years_before", 3))
YEARS_AFTER = int(window.get("years_after", 3))
eligibility = cfg.get("eligibility", {}) or {}

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
# ---- Settings: exploratory starting points, not decisions ----
SAVE_OUTPUTS = True
READ_NHDPLUS = True          # dam/station reach attributes for section 6 (slow first run, then cached)

MAD_THRESHOLD = 5.0          # |robust z| above this is an outlier flag
MAD_MIN_N = 10               # station-series with fewer values aren't screened
SNAP_LIMIT_M = 500           # station moved farther than this to reach a stream
SALINE_SC = 3000             # station median specific conductance (uS/cm) above this: brackish/saline
COLOCATED_M = 200            # stations on the same reach closer than this can be duplicates
RULE_MIN_SAMPLES, RULE_MIN_YEARS = 6, 2   # stricter than the current 3 samples / 2 years per side
RULE_MIN_SHARED_MONTHS = 3   # calendar months sampled both before and after removal
MIN_PAIRS_PER_SERIES = 20
DISTANCE_OFFSET_KM = 0.1     # log10(distance + offset) keeps 0 km defined

# Right-skewed, positive parameters are screened on log10 (as in analysis_eda.ipynb).
LOG_PARAMETERS = ["turbidity", "specific_conductance", "nitrate", "ammonia", "orthophosphate", "phosphorus", "tds"]
# Tighter than wq_rules.PLAUSIBLE_RANGES (which only drops impossible values): (low, high).
SUSPICIOUS = {"water_temp": (None, 35), "ph": (4, 11), "dissolved_oxygen": (None, 20),
              "do_saturation": (None, 200), "specific_conductance": (None, 10000), "turbidity": (None, 2000),
              "phosphorus": (None, 10), "orthophosphate": (None, 5), "nitrate": (None, 30),
              "ammonia": (None, 10), "tds": (None, 10000)}
ZERO_IMPLAUSIBLE = ["ph", "specific_conductance", "tds"]
NUTRIENTS = ["nitrate", "ammonia", "orthophosphate", "phosphorus"]   # a reported 0 should be censored
NO_UNIT_SLIP = ["ph", "water_temp", "do_saturation", "turbidity"]      # skipped by the x1000 check

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
con, VIEWS = au.connect(WAREHOUSE, YEARS_BEFORE, YEARS_AFTER,
                        int(eligibility.get("min_samples", DEFAULT_MIN_SAMPLES)),
                        int(eligibility.get("min_years", DEFAULT_MIN_YEARS)),
                        threads=os.environ.get("SLURM_CPUS_PER_TASK"), temp_dir=OUTPUT_DIR / "_duckdb_tmp")
HAS = set(VIEWS)


def q(sql):
    return con.sql(sql).df()


def save(frame, name):
    if SAVE_OUTPUTS:
        au.save_output(frame, name, OUTPUT_DIR)
    return frame


def sql_list(values):
    return ", ".join("'" + str(v) + "'" for v in values)


SERIES = au.series_sql()
LOG_LIST = sql_list(LOG_PARAMETERS)

# m: cleaned results; am: analysis rows (measurement x dam); pairs: one row per dam-station pair.
con.execute(f"CREATE TEMP TABLE m AS SELECT *, {SERIES} AS series, month(sample_date) AS month FROM wq_measurement")
con.execute(f"CREATE TEMP TABLE am AS SELECT *, {SERIES} AS series, AR_ID || ' | ' || station_id AS pair "
            "FROM analysis_measurement")
con.execute("""
    CREATE TEMP TABLE pairs AS
    SELECT AR_ID, station_id, pair, any_value(side) AS side, any_value(network_distance_km) AS network_distance_km,
           any_value(removal_year) AS removal_year, count(*) AS rows
    FROM am GROUP BY AR_ID, station_id, pair
""")
print(f"{cfg.name}: window -{YEARS_BEFORE}/+{YEARS_AFTER} years; reading {WAREHOUSE}")

## 1. How the joins expand the data

Each cleaned result is joined to **every dam linked to its station**, so one measurement can appear in several pairs. **How to read it:** `rows_per_measurement` above 1 is reuse, not duplication, as long as the copies agree (checked below the table). It still matters for the models: reused rows aren't independent observations.

In [ ]:
expansion = au.counts_table(con, {
    "wq_measurement (cleaned results)": "SELECT measurement_id, station_id, series FROM m",
    "x every linked dam with a removal year": f"SELECT measurement_id, station_id, AR_ID, {SERIES} AS series FROM measurement_window",
    "  ... pre/post years only": f"SELECT measurement_id, station_id, AR_ID, {SERIES} AS series FROM measurement_window WHERE period IN ('pre', 'post')",
    "analysis_measurement (has_both pair-series)": "SELECT measurement_id, station_id, AR_ID, series FROM am",
})
expansion["rows_per_measurement"] = (expansion["rows"] / expansion["measurements"]).round(2)
differ = con.sql("""SELECT count(*) FROM (SELECT measurement_id FROM am GROUP BY 1
                    HAVING count(DISTINCT result_value_std) > 1 OR count(DISTINCT sample_date) > 1
                        OR count(DISTINCT station_id) > 1)""").fetchone()[0]
print(f"Measurements whose copies disagree: {differ} (0 means every repeated row is reuse for another dam)")
expansion

## 2. Duplicates and repeated readings

The cleaning step already removed exact repeats and the same result filed under two activities at the same time (spec 3.6.5). What could remain:

| check | what it means |
|---|---|
| same result, different method ID | one result reported twice under two method codes |
| same station, day, series and value at a different time | likely a double report |
| same, but from NWIS and STORET | one agency re-publishing the other's data |
| co-located stations with identical values | two station IDs on the same reach within `COLOCATED_M`, same series, day and value: likely one sample under two IDs |

**How to read it:** `extra_rows` is how many rows a "keep one copy" rule would drop. Several readings on one day are **not** treated as duplicates; that is the second table.

In [ ]:
hav = au.haversine_m_sql("a.latitude", "a.longitude", "b.latitude", "b.longitude")
con.execute(f"""
    CREATE TEMP TABLE colocated AS
    SELECT x.measurement_id AS id_a, y.measurement_id AS id_b, x.provider AS provider_a, y.provider AS provider_b
    FROM wq_station a JOIN wq_station b ON a.NHDPlusID = b.NHDPlusID AND a.station_id < b.station_id AND {hav} <= {COLOCATED_M}
    JOIN m x ON x.station_id = a.station_id
    JOIN m y ON y.station_id = b.station_id AND y.sample_date = x.sample_date AND y.series = x.series
         AND y.result_value_std = x.result_value_std
""")
# Copies a "keep one" rule would drop: all but the lowest measurement_id per group; for
# co-located stations, the non-NWIS copy (else the second station's).
con.execute("""
    CREATE TEMP TABLE duplicate_copies AS
    SELECT measurement_id, 'same station/day/series/value' AS reason FROM (
        SELECT measurement_id, row_number() OVER (PARTITION BY station_id, sample_date, series, result_value_std
                                                  ORDER BY measurement_id) AS k
        FROM m) WHERE k > 1
    UNION ALL
    SELECT CASE WHEN provider_b = 'NWIS' AND provider_a <> 'NWIS' THEN id_a ELSE id_b END, 'co-located station copy'
    FROM colocated
""")
duplicates = q("""
    WITH a AS (SELECT count(*) AS n FROM m GROUP BY station_id, organization_id, activity_id, sample_date, sample_time,
                   parameter, sample_fraction, result_value_raw, result_unit_raw, detection_condition HAVING count(*) > 1),
    c AS (SELECT count(*) AS n, count(DISTINCT provider) AS providers FROM m
          GROUP BY station_id, sample_date, series, result_value_std HAVING count(*) > 1)
    SELECT 'same result, different method ID' AS check_name, count(*) AS groups, coalesce(sum(n) - count(*), 0) AS extra_rows FROM a
    UNION ALL SELECT 'same station/day/series/value, different time', count(*), coalesce(sum(n) - count(*), 0) FROM c
    UNION ALL SELECT '  ... of which across NWIS and STORET', count(*), coalesce(sum(n) - count(*), 0) FROM c WHERE providers > 1
    UNION ALL SELECT 'co-located stations with identical values', count(*), count(*) FROM colocated
""")
duplicates

**Several readings on the same day** per station and series. `depth_profile_days` have samples at several depths (lakes, deep pools). The other multi-reading days are repeated readings at one spot. `rows_if_daily` is what a hypothetical "one value per station, series and day" aggregation would leave. It isn't applied.

In [ ]:
same_day = q("""
    WITH g AS (SELECT series, count(*) AS n, count(DISTINCT depth_m) AS depths
               FROM m GROUP BY station_id, series, sample_date)
    SELECT series, sum(n) AS rows,
           round(100 * coalesce(sum(n) FILTER (WHERE n > 1), 0) / sum(n), 1) AS pct_rows_on_multi_reading_days,
           count(*) FILTER (WHERE n > 1 AND depths > 1) AS depth_profile_days,
           count(*) FILTER (WHERE n > 1 AND depths <= 1) AS other_multi_reading_days,
           count(*) AS rows_if_daily
    FROM g GROUP BY series ORDER BY rows DESC
""")
same_day

## 3. Values that passed cleaning but look wrong

Cleaning only drops impossible values (`wq_rules.PLAUSIBLE_RANGES`, e.g. pH 0–14). These flags catch values inside those limits that are still suspect:

- `zero`: exactly 0 where that can't happen (pH, conductance, TDS), or a nutrient reported as 0 instead of "below detection";
- `at_limit`: exactly on a `PLAUSIBLE_RANGES` limit (e.g. DO = 25), typical of instrument caps or entry codes;
- `suspicious`: beyond the tighter `SUSPICIOUS` limits in the settings;
- `x1000`: about 1000× (or 1/1000×) the station's median, a likely mg/µg (or µS/mS) slip;
- `unit_assumed`: share of nutrient values whose N/P basis was assumed (reported as bare mg/L), for context only.

In [ ]:
limits = pd.DataFrame([{"parameter_std": p, "plausible_low": lo, "plausible_high": hi,
                        "suspicious_low": SUSPICIOUS.get(p, (None, None))[0], "suspicious_high": SUSPICIOUS.get(p, (None, None))[1]}
                       for p, (lo, hi) in wq_rules.PLAUSIBLE_RANGES.items()])
limits[["suspicious_low", "suspicious_high"]] = limits[["suspicious_low", "suspicious_high"]].astype("float64")
con.register("limits", limits)
con.execute(f"""
    CREATE TEMP TABLE value_flags AS
    WITH s AS (SELECT station_id, series, median(result_value_std) AS med FROM m
               WHERE NOT is_censored AND result_value_std > 0 GROUP BY ALL HAVING count(*) >= {MAD_MIN_N})
    SELECT m.measurement_id, m.parameter_std, m.unit_basis_assumed,
           m.result_value_std = 0 AND (m.parameter_std IN ({sql_list(ZERO_IMPLAUSIBLE)})
               OR (m.parameter_std IN ({sql_list(NUTRIENTS)}) AND NOT m.is_censored)) AS f_zero,
           m.result_value_std = l.plausible_high OR (m.result_value_std = l.plausible_low AND l.plausible_low <> 0) AS f_at_limit,
           coalesce(m.result_value_std < l.suspicious_low OR m.result_value_std > l.suspicious_high, FALSE) AS f_suspicious,
           CASE WHEN m.parameter_std NOT IN ({sql_list(NO_UNIT_SLIP)}) AND m.result_value_std > 0 AND NOT m.is_censored
                THEN abs(abs(log10(m.result_value_std / s.med)) - 3) <= log10(1.5) END IS TRUE AS f_x1000
    FROM m JOIN limits l USING (parameter_std) LEFT JOIN s USING (station_id, series)
""")
q("""
    SELECT parameter_std, count(*) AS rows, sum(f_zero::INT) AS zero, sum(f_at_limit::INT) AS at_limit,
           sum(f_suspicious::INT) AS suspicious, sum(f_x1000::INT) AS x1000,
           round(100 * avg((f_zero OR f_at_limit OR f_suspicious OR f_x1000)::INT), 2) AS pct_any,
           round(100 * avg(unit_basis_assumed::INT), 1) AS pct_unit_assumed
    FROM value_flags GROUP BY 1 ORDER BY rows DESC
""")

**Related measurements that disagree**, within the same sampling visit (same station and `activity_id`; only uncensored values, one value per series per visit):

- a part larger than its whole: dissolved P > total P, orthophosphate > total P (beyond 10% and 0.005 mg/L);
- TDS / specific conductance outside 0.4–1.0 (typical is 0.55–0.75);
- reported DO saturation more than 15 points from the value computed from DO and water temperature (sea-level formula, so high sites read a little high).

Both measurements of a disagreeing visit are flagged.

In [ ]:
con.execute("""
    CREATE TEMP TABLE visit AS
    SELECT measurement_id, station_id, activity_id, series, result_value_std AS v FROM m
    WHERE NOT is_censored AND activity_id IS NOT NULL AND result_value_std IS NOT NULL
    QUALIFY count(*) OVER (PARTITION BY station_id, activity_id, series) = 1
""")
pair_check = lambda name, a, b, violation: f"""
    SELECT '{name}' AS check_name, x.measurement_id AS id_a, y.measurement_id AS id_b, NULL::BIGINT AS id_c, {violation} AS bad
    FROM visit x JOIN visit y ON x.station_id = y.station_id AND x.activity_id = y.activity_id
    WHERE x.series = '{a}' AND y.series = '{b}'"""
checks = [
    pair_check("dissolved P > total P", "phosphorus (Dissolved)", "phosphorus (Total)", "x.v > 1.1 * y.v AND x.v - y.v > 0.005"),
    pair_check("orthophosphate > total P", "orthophosphate (Dissolved)", "phosphorus (Total)", "x.v > 1.1 * y.v AND x.v - y.v > 0.005"),
    pair_check("TDS / conductance outside 0.4-1.0", "tds", "specific_conductance", "y.v > 0 AND (x.v / y.v < 0.4 OR x.v / y.v > 1.0)"),
    f"""SELECT 'DO saturation off by > 15 points', x.measurement_id, s.measurement_id, t.measurement_id,
               abs(s.v - 100 * x.v / {au.do_saturation_sql('t.v')}) > 15
        FROM visit x
        JOIN visit t ON t.station_id = x.station_id AND t.activity_id = x.activity_id AND t.series = 'water_temp'
        JOIN visit s ON s.station_id = x.station_id AND s.activity_id = x.activity_id AND s.series = 'do_saturation'
        WHERE x.series = 'dissolved_oxygen'""",
]
con.execute("CREATE TEMP TABLE consistency AS " + " UNION ALL ".join(checks))
con.execute("""CREATE TEMP TABLE inconsistent AS
               SELECT DISTINCT id FROM (SELECT unnest([id_a, id_b, id_c]) AS id FROM consistency WHERE bad)
               WHERE id IS NOT NULL""")
q("""SELECT check_name, count(*) AS visits_compared, count(*) FILTER (WHERE bad) AS disagree,
            round(100 * avg(bad::INT), 2) AS pct_disagree FROM consistency GROUP BY 1 ORDER BY 1""")

## 4. Outliers within each station

A robust z-score per station and series: `z = (x − median) / (1.4826 × MAD)`. It is comparable to an ordinary z-score but isn't pulled by the outliers it is looking for. Log10 is used for the skewed parameters. The median and MAD come from uncensored values only. Station-series with fewer than `MAD_MIN_N` values are not screened (`not_screened`). If more than half of a station's values are identical, MAD is 0, so the mean absolute deviation is used instead.

**How to read it:** if `pct_z>5` is much smaller than `pct_z>3.5` and close to `pct_z>7`, the values beyond 5 form a separate tail (likely errors) and the threshold choice matters little. Real storm or sediment-pulse values can also be flagged, which is why these are flags, not deletions.

In [ ]:
au.robust_z(con, f"""
    SELECT measurement_id, station_id, series, parameter_std, is_censored,
           CASE WHEN parameter_std IN ({LOG_LIST}) THEN CASE WHEN result_value_std > 0 THEN log10(result_value_std) END
                ELSE result_value_std END AS x
    FROM m""", ["station_id", "series"], "robust", MAD_MIN_N, reference_filter="NOT is_censored")
q(f"""
    SELECT parameter_std, count(*) AS rows,
           count(*) FILTER (WHERE robust_z IS NULL) AS not_screened,
           round(100 * avg((abs(robust_z) > 3.5)::INT), 2) AS "pct_z>3.5",
           round(100 * avg((abs(robust_z) > {MAD_THRESHOLD})::INT), 2) AS "pct_z>{MAD_THRESHOLD:g}",
           round(100 * avg((abs(robust_z) > 7)::INT), 2) AS "pct_z>7",
           count(DISTINCT station_id) FILTER (WHERE abs(robust_z) > {MAD_THRESHOLD}) AS stations_flagged
    FROM robust GROUP BY 1 ORDER BY rows DESC
""")

In [ ]:
# All measurement-level flags in one table (saved): one row per cleaned result.
measurement_flags = q(f"""
    SELECT m.measurement_id, m.station_id, m.series,
           v.f_zero, v.f_at_limit, v.f_suspicious, v.f_x1000,
           m.measurement_id IN (SELECT id FROM inconsistent) AS f_inconsistent,
           coalesce(abs(r.robust_z) > {MAD_THRESHOLD}, FALSE) AS f_outlier, r.robust_z,
           m.measurement_id IN (SELECT measurement_id FROM duplicate_copies) AS f_duplicate_copy
    FROM m JOIN value_flags v USING (measurement_id) LEFT JOIN robust r USING (measurement_id)
""")
con.register("measurement_flags", measurement_flags)
save(measurement_flags, "measurement_flags.parquet")
flag_cols = [c for c in measurement_flags.columns if c.startswith("f_")]
print("Share of cleaned results flagged (%):")
(100 * measurement_flags[flag_cols].mean()).round(3).to_frame("pct").T

## 5. Data that may not belong in the analysis

**Stations**, using what the warehouse already stores:
- `saline`: median specific conductance above `SALINE_SC` (fresh water is usually well under 1,500 µS/cm);
- `non_riverine`: the name suggests a marsh, bay, estuary, lake, pond, reservoir, slough, lagoon or canal, or the NHDPlus reach is a canal, connector or coastline;
- `far_snap`: moved more than `SNAP_LIMIT_M` to reach a stream, so it may be on the wrong reach;
- `bad_drainage_area`: no upstream area, the NHDPlus −9999 sentinel, or a truncated watershed;
- `name_mismatch` (context only): the station's name doesn't mention its reach's river. Weak on its own, strong together with `far_snap`.

**Pairs and series:**
- `overlap`: another removal linked to the same station falls inside this pair's window, so a change can't be attributed to one dam;
- `thin`: fewer than `RULE_MIN_SAMPLES` samples or `RULE_MIN_YEARS` years on a side;
- `few_shared_months`: fewer than `RULE_MIN_SHARED_MONTHS` calendar months sampled both before and after, so the comparison mostly measures season;
- `small_series`: the series has fewer than `MIN_PAIRS_PER_SERIES` pairs in total.

In [ ]:
WATER_BODY = r"\b(?:marsh|estuar\w*|bay|tidal|lake|pond|reservoir|impoundment|harbou?r|slough|lagoon|bayou|canal|ditch)\b"
stations = q(f"""
    WITH sc AS (SELECT station_id, median(result_value_std) AS sc_median FROM m
                WHERE parameter_std = 'specific_conductance' GROUP BY 1)
    SELECT s.station_id, s.station_name, s.snap_distance_m, r.gnis_name AS reach_name, r.feature_type,
           sc.sc_median, w.area_km2, w.nhdplus_total_area_km2, w.truncated
    FROM (SELECT DISTINCT station_id FROM pairs) p JOIN wq_station s USING (station_id)
    LEFT JOIN streams r ON r.nhdplus_id = s.NHDPlusID
    LEFT JOIN sc USING (station_id) LEFT JOIN station_watershed w USING (station_id)
""")
stations["saline"] = stations["sc_median"] > SALINE_SC
stations["non_riverine"] = (stations["station_name"].fillna("").str.lower().str.contains(WATER_BODY, regex=True)
                            | stations["feature_type"].isin([336, 334, 566]))
stations["far_snap"] = stations["snap_distance_m"] > SNAP_LIMIT_M
stations["bad_drainage_area"] = (stations["area_km2"].isna() | ~(stations["nhdplus_total_area_km2"] > 0)
                                 | stations["truncated"].fillna(False).astype(bool))
stations["name_mismatch"] = [au.name_matches(n, r) is False for n, r in zip(stations["station_name"], stations["reach_name"])]
STATION_FLAGS = ["saline", "non_riverine", "far_snap", "bad_drainage_area", "name_mismatch"]
con.register("station_flags", stations[["station_id"] + STATION_FLAGS])

con.execute(f"""
    CREATE TEMP TABLE pair_series AS
    SELECT pair, series,
           least(count(*) FILTER (WHERE period = 'pre'), count(*) FILTER (WHERE period = 'post')) AS min_samples,
           least(count(DISTINCT year) FILTER (WHERE period = 'pre'), count(DISTINCT year) FILTER (WHERE period = 'post')) AS min_years,
           coalesce(len(list_intersect(list(DISTINCT month) FILTER (WHERE period = 'pre'),
                                       list(DISTINCT month) FILTER (WHERE period = 'post'))), 0) AS shared_months
    FROM am GROUP BY pair, series
""")
con.execute(f"""
    CREATE TEMP TABLE pair_overlap AS
    SELECT p.pair, count(o.AR_ID) > 0 AS overlap, string_agg(o.AR_ID, ', ') AS overlapping_dams
    FROM pairs p
    LEFT JOIN (SELECT l.station_id, l.AR_ID, d.Year_Removed FROM dam_station_link l JOIN dam_removal d USING (AR_ID)) o
      ON o.station_id = p.station_id AND o.AR_ID <> p.AR_ID
     AND o.Year_Removed BETWEEN p.removal_year - {YEARS_BEFORE} AND p.removal_year + {YEARS_AFTER}
    GROUP BY p.pair
""")
small = q(f"SELECT series FROM am GROUP BY series HAVING count(DISTINCT pair) < {MIN_PAIRS_PER_SERIES}")["series"]
print("Small series:", ", ".join(small) or "none")

In [ ]:
# One row per analysis row with every flag (measurement, station, pair, covariate); used here and in section 8.
con.execute(f"""
    CREATE TEMP TABLE am_flags AS
    SELECT a.measurement_id, a.AR_ID, a.station_id, a.pair, a.series,
           f.f_duplicate_copy, f.f_zero OR f.f_at_limit OR f.f_suspicious AS f_bad_value, f.f_x1000,
           f.f_inconsistent, f.f_outlier,
           s.saline, s.non_riverine, s.far_snap, s.bad_drainage_area, s.name_mismatch,
           o.overlap,
           ps.min_samples < {RULE_MIN_SAMPLES} OR ps.min_years < {RULE_MIN_YEARS} AS thin,
           ps.shared_months < {RULE_MIN_SHARED_MONTHS} AS few_shared_months,
           a.series IN ({sql_list(small) or "''"}) AS small_series,
           a.air_temp_mean_c IS NULL OR a.precip_prev_month_mm IS NULL AS no_climate,
           a.pct_forest IS NULL OR a.land_cover_year >= a.removal_year AS no_pre_removal_land_cover,
           a.dam_height_ft IS NULL AS no_dam_height
    FROM am a JOIN measurement_flags f ON f.measurement_id = a.measurement_id
    JOIN station_flags s ON s.station_id = a.station_id JOIN pair_overlap o ON o.pair = a.pair
    JOIN pair_series ps ON ps.pair = a.pair AND ps.series = a.series
""")
total = con.sql("SELECT count(*) FROM am_flags").fetchone()[0]
reach = pd.concat([q(f"""SELECT '{f}' AS flag, count(DISTINCT station_id) AS stations, count(DISTINCT pair) AS pairs,
                         count(DISTINCT AR_ID) AS dams, round(100 * count(*) / {total}, 1) AS pct_rows
                         FROM am_flags WHERE {f}""")
                   for f in STATION_FLAGS + ["overlap", "thin", "few_shared_months", "small_series",
                                             "no_climate", "no_pre_removal_land_cover", "no_dam_height"]],
                  ignore_index=True)
reach

In [ ]:
print("Examples: flagged stations")
stations[stations[["saline", "non_riverine", "far_snap"]].any(axis=1)][
    ["station_id", "station_name", "reach_name", "sc_median", "snap_distance_m"]].head(10)

## 6. Distance and drainage-area ratio

Distance is kept as it is; `log10(distance + 0.1 km)` is added for modeling, and no near/far cutoff is introduced. The **drainage-area ratio** (station upstream area ÷ dam upstream area) measures how much other water has joined by the time the river reaches the station: about 1 means the station sees mostly the dam's water; 50 means the dam's water is a small share. It is a proxy for mixing, not a measurement. Both areas are NHDPlus `TotDASqKm` at each reach's downstream end, read **read-only** from the NHDPlus files already on disk (the warehouse doesn't store the dam's area). The first run scans each package (slow on `/courses`); the result is cached.

For **upstream** stations, the same files give the channel slope, which supports a rough screen of whether the station sat in the former reservoir: a dam of height *H* backs water up roughly *H* ÷ slope. Such a station is then affected by the removal itself, rather than being a control.

In [ ]:
cache = OUTPUT_DIR / "nhdplus_reach_attributes.parquet"
if not READ_NHDPLUS:
    vaa = pd.DataFrame({"nhdplusid": pd.Series(dtype="Int64"), "totdasqkm": [], "slope": []})
elif cache.exists():
    vaa = pd.read_parquet(cache)
else:
    ids = q("""SELECT NHDPlusID FROM dam_removal WHERE AR_ID IN (SELECT AR_ID FROM pairs)
               UNION SELECT NHDPlusID FROM wq_station WHERE station_id IN (SELECT station_id FROM pairs)""")["NHDPlusID"]
    huc8s = q("""SELECT HUC8 AS h FROM dam_removal WHERE AR_ID IN (SELECT AR_ID FROM pairs)
                 UNION SELECT huc8 FROM wq_station WHERE station_id IN (SELECT station_id FROM pairs)""")["h"]
    packages = set()
    for h in huc8s.dropna():
        try:
            packages.add(find_huc8_geodatabase(DATA_ROOT / "data" / "processed" / "nhdplus_extracted", h))
        except FileNotFoundError:
            pass
    vaa, notes = au.read_vaa(sorted(packages), ids, ["NHDPlusID", "TotDASqKm", "Slope"], progress=True)
    save(vaa, cache.name)
con.register("vaa", vaa)

features = q(f"""
    SELECT p.pair, p.AR_ID, p.station_id, p.side, p.network_distance_km,
           log10(p.network_distance_km + {DISTANCE_OFFSET_KM}) AS log10_distance_km,
           CASE WHEN vd.totdasqkm > 0 THEN vd.totdasqkm END AS dam_area_km2,
           CASE WHEN vs.totdasqkm > 0 THEN vs.totdasqkm END AS station_area_km2,
           d.Dam_Height_ft * 0.3048 / nullif(CASE WHEN vd.slope > 0 THEN vd.slope END, 0) / 1000 AS backwater_km
    FROM pairs p JOIN dam_removal d USING (AR_ID) JOIN wq_station s ON s.station_id = p.station_id
    LEFT JOIN vaa vd ON vd.nhdplusid = d.NHDPlusID LEFT JOIN vaa vs ON vs.nhdplusid = s.NHDPlusID
""")
features["drainage_area_ratio"] = features["station_area_km2"] / features["dam_area_km2"]
r = features["drainage_area_ratio"]
features["ratio_status"] = np.select(
    [features["dam_area_km2"].isna(), features["station_area_km2"].isna(),
     ((features["side"] == "downstream") & (r < 0.99)) | ((features["side"] == "upstream") & (r > 1.01))],
    ["no dam area", "no station area", "direction inconsistent"], "ok")
features["upstream_class"] = np.where(features["side"] != "upstream", None, np.select(
    [features["backwater_km"].isna(), features["network_distance_km"] <= features["backwater_km"]],
    ["no estimate (height or slope missing)", "possibly in former reservoir"], "above estimated reservoir"))
features = features.merge(q("SELECT pair, overlap, overlapping_dams FROM pair_overlap"), on="pair")
save(features, "pair_features.parquet")
display(pd.crosstab(features["ratio_status"], features["side"]))
display(features.groupby("side")["drainage_area_ratio"].describe(percentiles=[0.1, 0.5, 0.9]).round(1))
features["upstream_class"].value_counts().to_frame("upstream pairs")

In [ ]:
ok = features[features["ratio_status"] == "ok"]
ax = sns.scatterplot(data=ok, x="log10_distance_km", y=np.log10(ok["drainage_area_ratio"]), hue="side", s=12, alpha=0.6)
ax.set(xlabel="log10(network distance km + 0.1)", ylabel="log10 drainage-area ratio (0 = same area)",
       title="Distance vs drainage-area ratio, one point per pair")
plt.show()

## 7. Sampling imbalance

The share of **rows**, **pairs** and **dams** in each category. **How to read it:** a row-weighted model sees `pct_rows`; when that is far above `pct_pairs`, a few heavily sampled pairs dominate. `pct_dams` counts dams with any rows in the category, so it doesn't sum to 100. Nothing is reweighted here.

In [ ]:
con.execute("""
    CREATE TEMP TABLE dims AS
    SELECT a.pair, a.AR_ID, a.provider, a.side, a.period, a.season, substr(d.HUC8, 1, 2) AS huc2_region,
           CAST(CAST(floor(a.removal_year / 10) * 10 AS INTEGER) AS VARCHAR) || 's' AS removal_decade
    FROM am a JOIN dam_removal d USING (AR_ID)
""")
n = q("SELECT count(*) AS rows, count(DISTINCT pair) AS pairs, count(DISTINCT AR_ID) AS dams FROM dims").iloc[0]
imbalance = pd.concat([q(f"""
    SELECT '{dim}' AS dimension, CAST({dim} AS VARCHAR) AS category,
           round(100 * count(*) / {n['rows']}, 1) AS pct_rows,
           round(100 * count(DISTINCT pair) / {n['pairs']}, 1) AS pct_pairs,
           round(100 * count(DISTINCT AR_ID) / {n['dams']}, 1) AS pct_dams,
           round(count(*) / count(DISTINCT pair)) AS rows_per_pair
    FROM dims GROUP BY {dim} ORDER BY category""")
    for dim in ["provider", "side", "period", "season", "huc2_region", "removal_decade"]], ignore_index=True)
imbalance

In [ ]:
# Concentration: share of rows from the most-sampled stations, dams and pairs.
rows = []
fig, ax = plt.subplots(figsize=(6, 4))
for unit in ["station_id", "AR_ID", "pair"]:
    counts = q(f"SELECT count(*) AS n FROM am GROUP BY {unit} ORDER BY n DESC")["n"]
    share = counts.cumsum() / counts.sum()
    rows.append({"unit": unit, "units": len(counts),
                 **{f"top {t:.0%}": round(100 * share.iloc[max(1, round(t * len(counts))) - 1], 1) for t in [0.01, 0.05, 0.10]}})
    ax.plot(np.r_[0, np.arange(1, len(counts) + 1) / len(counts)], np.r_[0, share], label=unit)
ax.plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
ax.set(xlabel="share of units, most-sampled first", ylabel="share of rows", title="Concentration of analysis rows")
ax.legend()
plt.show()
print("% of analysis rows from the top 1/5/10% of units:")
pd.DataFrame(rows)

## 8. What each rule would remove (hypothetical; nothing applied)

The flags above as candidate exclusions, in order. `pct_rows_alone` is the share of rows each rule touches on its own. The `remaining_*` columns are what would be left after applying it **together with all rules above it**. Water-quality rules come first; the three covariate rules at the end only matter for models that need climate, land cover or dam height. `name_mismatch` and `direction inconsistent` (section 6) are left out: they are context, not exclusions.

In [ ]:
RULES = [
    ("duplicate copy (keep one)", "f_duplicate_copy"),
    ("zero / at limit / suspicious value", "f_bad_value"),
    ("about 1000x station median", "f_x1000"),
    ("inconsistent with a related measurement", "f_inconsistent"),
    (f"outlier |z| > {MAD_THRESHOLD:g}", "f_outlier"),
    (f"series with < {MIN_PAIRS_PER_SERIES} pairs", "small_series"),
    (f"< {RULE_MIN_SAMPLES} samples or < {RULE_MIN_YEARS} years on a side", "thin"),
    (f"< {RULE_MIN_SHARED_MONTHS} shared pre/post months", "few_shared_months"),
    ("another removal inside the window", "overlap"),
    ("saline station", "saline"),
    ("non-riverine station", "non_riverine"),
    (f"snapped > {SNAP_LIMIT_M} m", "far_snap"),
    ("drainage area missing, invalid or truncated", "bad_drainage_area"),
    ("covariates: no climate", "no_climate"),
    ("covariates: no pre-removal land cover", "no_pre_removal_land_cover"),
    ("covariates: no dam height", "no_dam_height"),
]
COUNTS = ("count(*) AS rows, count(DISTINCT measurement_id) AS measurements, count(DISTINCT station_id) AS stations, "
          "count(DISTINCT pair) AS pairs, count(DISTINCT AR_ID) AS dams")
records = [{"rule": "analysis_measurement as built", "pct_rows_alone": 0.0,
            **q(f"SELECT {COUNTS} FROM am_flags").add_prefix("remaining_").iloc[0].to_dict()}]
applied = []
for label, column in RULES:
    applied.append(f"coalesce({column}, FALSE)")
    alone = con.sql(f"SELECT 100 * avg(coalesce({column}, FALSE)::INT) FROM am_flags").fetchone()[0]
    remaining = q(f"SELECT {COUNTS} FROM am_flags WHERE NOT ({' OR '.join(applied)})").add_prefix("remaining_")
    records.append({"rule": label, "pct_rows_alone": round(alone, 2), **remaining.iloc[0].to_dict()})
rule_impact = pd.DataFrame(records)
save(rule_impact, "rule_impact.csv")
rule_impact

### Decisions for the model-specific stage

Use the table above to decide:

1. **Duplicates** (section 2): keep one copy? Collapse same-day readings to one per day, or per depth?
2. **Values and outliers** (sections 3–4): which flags become exclusions, and at what robust-z threshold?
3. **What belongs** (section 5): which station rules to apply; how to handle overlapping removals (exclude, nearest dam only, or keep and cluster by station); the minimum-data and shared-month rules; dropping small series.
4. **Features** (section 6): use log distance and the drainage-area ratio, and treat upstream stations that were possibly in the former reservoir as treated sites rather than controls?
5. **Imbalance** (section 7): row-, pair- or dam-level weighting or aggregation, separately for the mixed-effects model and XGBoost.